# 재정렬기를 공동인용으로 미세조정 (Colab / RunPod)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndigkwk/arxiv_graph_RAG/blob/main/notebooks/rerank_finetune_colab.ipynb)

위 배지를 누르면 이 노트북이 Colab에서 바로 열립니다. **런타임 > 런타임 유형 변경**에서
GPU(L4 또는 A100)를 고르고 위에서 아래로 실행하면 됩니다. 고칠 셀은 **1절 하나**입니다.

**RunPod에서도 그대로 돕니다.** JupyterLab에 이 파일을 올리고 같은 순서로 실행하세요 —
노트북이 환경을 보고 저장 위치를 고릅니다(Colab은 Drive, RunPod은 `/workspace`
네트워크 볼륨). 토큰은 Colab 시크릿·환경변수·입력 중 있는 것을 씁니다.

## 넘어야 하는 바 — 돌리기 전에 적어 둡니다

[ADR-057](https://github.com/rndigkwk/arxiv_graph_RAG/blob/main/docs/adr/ADR-057-fine-tune-the-reranker-on-cocitation.md)이
정한 선입니다. **여기 적어 두는 이유는 결과를 보고 바를 움직일 수 없게 하는 것**입니다 —
ADR-039·050에서 그렇게 해서 판정이 성립했습니다.

미래 라벨 과제의 **간선 0개 집단(1,112 시드), 후보 풀 50**에서 코사인을 **쌍체 구간이 0을
제외하도록** 이겨야 합니다.

| 지표 | 코사인 | 사전학습 재정렬기 | 미세조정이 넘어야 하는 선 |
|---|---:|---:|---|
| recall@5 | **0.1227** | 0.0868 | > 0.1227, 구간 0 제외 |
| hit@5 | **0.4739** | 0.3723 | > 0.4739, 구간 0 제외 |
| MRR | **0.3457** | 0.2523 | > 0.3457, 구간 0 제외 |

- 사전학습 판은 recall@5에서 **0.0359 낮습니다.** 그것을 메우고 **그 위로** 가야 합니다
  (41% 상대 개선). **동점은 통과가 아닙니다** — 코사인이 공짜입니다.
- 풀 50의 **상한이 0.2690**입니다. recall@5가 그보다 크게 나오면 **측정이 틀린 것**입니다.
- 간선 1~4개 집단(251 시드)을 깨뜨리면 전체로는 쓸 수 없습니다.

**판정은 이 노트북에서 하지 않습니다.** 여기서는 점수만 내고, 순위와 구간은 저장소가 냅니다 —
동점 처리까지 다른 변형과 같아야 비교가 성립합니다.

## 무엇을 학습하는가 — 추론이 보는 것과 같게

저장소가 보낸 학습 쌍은 **시드마다 `풀 ∩ 공동인용`을 1, 나머지 풀을 0**으로 둡니다.
풀 밖 양성은 들어 있지 않습니다 — 추론에서 볼 일이 없는 쌍이기 때문입니다(실측: 풀 49편에
양성 p50 **2**개, 비율 약 1:12).

그래프를 **2026-07**로 잘라 만들었습니다. 평가는 2026-08로 자르므로 **학습 라벨과 평가
정답이 구조적으로 분리**됩니다(내보내기 매니페스트의 `leak_check`가 0을 적어 둡니다).


## 1. 설정 — 여기만 바꿉니다

In [ ]:
# 이 셀만 바꾸고 나머지는 그대로 실행하세요.

# **이 모델이어야 합니다.** 같은 계열 셋 중 1024를 받는 것은 v2-m3뿐입니다
# (2026-10-20 실측, AutoConfig.max_position_embeddings):
#   bge-reranker-base    512   <- 1024를 주면 장치 어서션으로 터집니다
#   bge-reranker-large   512   <- 큰데도 512입니다
#   bge-reranker-v2-m3  8192   <- 이것
MODEL = "BAAI/bge-reranker-v2-m3"

# 쌍 길이. 이 말뭉치 실측은 중앙값 636, p95 850, 최대 1025였습니다.
#   512로 자르면 쌍의 84.8%가 잘립니다 -> 재는 것이 모델이 아니라 자른 자리가 됩니다
MAX_SEQ = 1024

# **학습은 채점보다 무겁습니다.** 역전파가 활성값을 들고 있어야 하므로 같은 배치에서
# VRAM이 몇 배입니다. **16은 OOM이었고 8로 완주했습니다**(2026-10-22 실측, MAX_SEQ=1024).
# 그래서 8에서 시작합니다 - OOM이 나면 자동으로 절반씩 줄입니다.
BATCH = 8

# 1에폭부터 봅니다. ADR-057이 적은 선: **한 에폭 뒤 손실이 내려가지 않으면 그것이 답이고,
# 더 돌리지 않습니다.** 늘리는 것은 결과를 보고 정합니다.
EPOCHS = 1
LR = 2e-5
WARMUP_RATIO = 0.1

# 검증에 뗄 시드 비율. 손실이 내려가는지 **학습 중에** 보려면 필요합니다.
VAL_RATIO = 0.1

# 저장소가 보낸 파일 넷. **HF 데이터셋에 올라가 있으면 그대로 받아 옵니다.**
TRAIN_PAIRS = "rerank_training_pairs.jsonl.gz"
TRAIN_TEXTS = "rerank_training_texts.jsonl.gz"
EVAL_PAIRS = "rerank_pairs.jsonl.gz"
EVAL_TEXTS = "rerank_texts.jsonl.gz"

USE_HF = True
DATA_REPO = "arxiv-graphrag-colab"

# 학습한 가중치도 HF에 올릴지. 끄면 점수만 올립니다(파일이 2.3GB입니다).
UPLOAD_WEIGHTS = False

# SMOKE=True면 시드 40개로 학습하고 40개만 채점해 코드가 도는지 봅니다.
SMOKE = False

# 학습 과정을 **어디에 남길지.** 실행 중인 VM 안에만 쓰면 런타임이 끊길 때 기록이 아니라
# 착각이 됩니다. mlflow는 원격 서버에 바로 쌓입니다.
TRACKING = "mlflow"          # "mlflow" | "none"

# DagsHub는 저장소마다 MLflow 서버가 하나입니다. URI와 계정명은 **비밀이 아니라** 적어
# 두고, 토큰만 실행할 때 입력받습니다 - 노트북에 토큰을 적으면 커밋에 남습니다.
MLFLOW_TRACKING_URI = "https://dagshub.com/rndigkwk/arxiv_graph_RAG.mlflow"
MLFLOW_USERNAME = "rndigkwk"
MLFLOW_EXPERIMENT = "arxiv-graphrag-reranker"

# 중간 곡선을 몇 스텝마다 찍을지. 0이면 학습 끝에 한 번만 잽니다.
EVAL_STEPS = 200

# ── 재개 ─────────────────────────────────────────────────────────────────
# **3.7시간짜리 구간에 재개가 없으면 끊길 때 전부 날아갑니다.** 이 프로젝트가 이미 한 번
# 겪었습니다(finetune_embedding_colab.ipynb: "중간에 커널이 죽으면 전부 날아갑니다.
# 실제로 한 번 겪었습니다"). 2026-10-22 실측으로 학습 4.75쌍/초이니 63,899쌍이 약 3.7시간
# 입니다.
#
# 학습이 끝나면 **먼저 저장합니다.** 그러면 채점 도중에 끊겨도 학습을 다시 하지 않습니다.
SAVE_DIR = "finetuned"
RESUME_FROM_SAVED = True

# 채점은 **조각 파일**로 씁니다. 끊겼다 다시 돌리면 이미 끝난 조각을 건너뜁니다 -
# 사전학습 노트북(rerank_pairs_colab)이 쓰는 것과 같은 방식입니다.
CHUNK_SEEDS = 100

# 저장과 조각을 **끊겨도 남는 자리**에 둡니다. 어디가 그 자리인지는 환경마다 다릅니다.
#   Colab  : Drive를 붙여야 남습니다(로컬 디스크는 런타임과 함께 사라집니다)
#   RunPod : 네트워크 볼륨(보통 /workspace)이 이미 그 자리입니다 - 붙일 것이 없습니다
# PERSIST=True면 아래 셀이 **환경을 보고** 알아서 고릅니다.
PERSIST = True
COLAB_DIR = "/content/drive/MyDrive/arxiv-graphrag-reranker"
RUNPOD_DIR = "/workspace/arxiv-graphrag-reranker"


## 2. 환경 확인

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    print('GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고르세요.')
    if not SMOKE:
        raise SystemExit('GPU 없이 학습을 시작하지 않습니다.')
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  {total:.1f}GB')

    # **CUDA가 이미 망가져 있지 않은지 먼저 봅니다.** 장치 어서션(cudaErrorAssert)이 한 번
    # 나면 그 프로세스의 CUDA 컨텍스트가 끝까지 망가집니다. 그 뒤의 모든 CUDA 호출이
    # 실패하는데 보통 모델 로드에서 터져서 **원인처럼 보입니다** - 2026-10-20에 정확히
    # 그 일이 있었습니다. 작은 연산 하나로 몇 초 만에 가려냅니다.
    try:
        torch.zeros(8, device='cuda').sum().item()
        print('CUDA 상태 정상')
    except Exception as exc:
        raise SystemExit(
            f'CUDA가 이미 망가진 상태입니다({type(exc).__name__}). **런타임 > 세션 다시 '
            f'시작**을 누르고 1절부터 다시 돌리세요. 원문: {str(exc).splitlines()[0][:150]}'
        ) from None

    print(f'배치 {BATCH}, 최대 길이 {MAX_SEQ}로 시작합니다. OOM이 나면 절반씩 줄입니다.')

In [ ]:
%pip -q install -U sentence-transformers datasets accelerate

# **설치가 됐는지 여기서 확인합니다.** pip는 의존성이 어긋나도 경고만 하고 넘어갑니다 -
# 2026-10-20에 huggingface_hub를 올렸다가 sentence-transformers와 어긋나 학습 직전에
# 터졌습니다.
#
# **그리고 "임포트가 되는가"만 보면 부족합니다.** 2026-10-22에 그 자리에서 또 걸렸습니다 -
# `CrossEncoder`와 `InputExample`은 멀쩡히 임포트됐는데, `fit()`이 **호출 시점에**
# `datasets.Dataset`을 쓰고 그 패키지가 없어서 `NameError: name 'Dataset' is not defined`
# 로 죽었습니다(sentence_transformers/cross_encoder/fit_mixin.py). 최신 판의 `fit`은 구
# API를 `CrossEncoderTrainer` 경로로 바꿔 주는 셸이고, 그 변환이 `datasets`를 요구합니다.
#
# 그래서 **그 심볼까지 지금 확인합니다.** 세 시간짜리 학습이 시작하자마자 죽는 것보다
# 여기서 한 줄이 낫습니다.
_missing = []
try:
    import sentence_transformers
    from sentence_transformers import CrossEncoder, InputExample  # noqa: F401
    print('sentence-transformers', sentence_transformers.__version__)
except Exception as exc:
    raise SystemExit(
        f'설치가 어긋났습니다({type(exc).__name__}: {exc}). **런타임 > 세션 다시 시작**을 '
        '누른 뒤 1절부터 다시 돌리세요.'
    ) from None

for _label, _probe in (
    # `fit()`이 내부에서 쓰는 바로 그 심볼입니다.
    ('datasets', 'from datasets import Dataset'),
    # 그 경로가 쓰는 transformers Trainer가 요구합니다.
    ('accelerate', 'import accelerate'),
):
    try:
        exec(_probe)
    except Exception as exc:
        _missing.append(f'{_label} ({type(exc).__name__})')

if _missing:
    raise SystemExit(
        '학습 경로가 쓰는 패키지가 없습니다: ' + ', '.join(_missing) + '.\n'
        'pip install -U datasets accelerate 를 돌리고 **런타임 > 세션 다시 시작** 뒤 '
        '1절부터 다시 돌리세요.\n'
        '(fit()은 임포트 때가 아니라 **호출 때** 이것들을 씁니다 - 그래서 위 임포트만으로는 '
        '드러나지 않습니다.)'
    )
print('datasets·accelerate 확인')


### 길이 한도 확인 — 가중치를 받기 전에 막습니다

In [ ]:
from transformers import AutoConfig

# 설정만 읽습니다(몇 KB). `MAX_SEQ`가 한도를 넘으면 2.3GB를 받을 이유가 없습니다 -
# 넘긴 채 돌리면 장치 어서션으로 죽고 **오류에 길이 이야기가 한 줄도 없습니다.**
_cfg = AutoConfig.from_pretrained(MODEL)
_limit = getattr(_cfg, 'max_position_embeddings', None)
print(f'{MODEL}: max_position_embeddings = {_limit}')
if _limit and MAX_SEQ > _limit - 2:
    raise SystemExit(
        f'MAX_SEQ={MAX_SEQ}가 이 모델의 한도({_limit})를 넘습니다. 1절에서 MAX_SEQ를 '
        f'{_limit - 2} 이하로 줄이거나 MODEL을 v2-m3로 두세요.'
    )

## 3. 허깅페이스 토큰 — **여기서 한 번만** 묻습니다

In [ ]:
import getpass
import os

# 토큰을 세 자리에서 찾습니다 - 환경변수(RunPod 팟 설정), Colab 시크릿, 그리고 입력.
HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    print('환경변수 HF_TOKEN을 씁니다.')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print('Colab 시크릿의 HF_TOKEN을 씁니다.')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF 토큰(write 권한 필요): ')

if not HF_TOKEN:
    raise SystemExit('토큰이 비었습니다. 4절의 받기와 7절의 올리기에 필요합니다.')

# 누구의 토큰인지만 확인합니다. 토큰 자체는 출력하지 않습니다.
from huggingface_hub import HfApi

HF_OWNER = HfApi(token=HF_TOKEN).whoami()['name']
print('계정:', HF_OWNER)


### DagsHub 토큰 — 학습 전에 **여기서** 묻습니다

MLflow 서버가 DagsHub에 있습니다. **학습 전에 묻는 이유**가 HF 토큰과 같습니다 — 한 시간
학습이 끝난 뒤 입력을 기다리며 멈춰 있지 않게 하려는 것입니다.

`TRACKING = "none"`으로 두면 이 셀은 아무것도 묻지 않고, 그때는 **기록이 Colab 출력에만
남습니다**(런타임이 끊기면 사라집니다).


In [ ]:
import os

if TRACKING == "mlflow":
    if not MLFLOW_TRACKING_URI or not MLFLOW_USERNAME:
        raise SystemExit(
            "TRACKING='mlflow'인데 URI 또는 계정명이 비어 있습니다. 원격 서버를 가리키게 "
            "두세요 - mlruns/를 이 런타임에 쓰면 런타임과 함께 사라집니다."
        )
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
    if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
        _dags = os.environ.get('DAGSHUB_TOKEN')
        if not _dags:
            try:
                from google.colab import userdata
                _dags = userdata.get('DAGSHUB_TOKEN')
                if _dags:
                    print('Colab 시크릿의 DAGSHUB_TOKEN을 씁니다.')
            except Exception:
                pass
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _dags or getpass.getpass('DagsHub 토큰: ')
    if not os.environ["MLFLOW_TRACKING_PASSWORD"]:
        raise SystemExit('DagsHub 토큰이 비었습니다. TRACKING="none"으로 두면 건너뜁니다.')
    # **`mlflow`가 아니라 `mlflow-skinny`입니다.** 전체 mlflow는 서버·UI까지 끌어오고,
    # 그 과정에서 `Flask<4` -> `blinker>=1.9`가 필요해집니다. OS 패키지로 깔린
    # blinker는 RECORD 파일이 없어 pip이 **지우기를 거부**합니다 - 2026-10-22에 RunPod
    # 팟에서 그 자리에 걸렸습니다(`uninstall-no-record-file`).
    #
    # 우리는 **추적 클라이언트만** 씁니다(set_tracking_uri·set_experiment·start_run·
    # log_params·log_metrics·log_artifact). 전부 skinny에 있고 Flask는 필요 없습니다.
    %pip -q install -U mlflow-skinny
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    # 토큰은 찍지 않습니다. 서버에 닿는지만 봅니다.
    print('MLflow 실험:', MLFLOW_EXPERIMENT, '@', MLFLOW_TRACKING_URI)
else:
    mlflow = None
    print('추적을 끕니다. 기록은 이 출력에만 남습니다.')


### 작업 폴더 — 끊겨도 남는 자리

`USE_DRIVE = True`면 Drive를 붙여 거기에 저장과 조각을 둡니다. **런타임이 회수돼도
남습니다.** 로컬에 두면 재접속까지만 버팁니다.


In [ ]:
import os
from pathlib import Path

# **환경을 보고 고릅니다.** `from google.colab import drive`를 조건 없이 부르면
# RunPod에서 ImportError로 즉시 터집니다 - 2026-10-22에 그 상태로 올렸습니다.
ON_COLAB = False
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    pass

if not PERSIST:
    WORK = Path('.')
    print('PERSIST=False - 이 런타임 안에만 씁니다. 끊기면 사라집니다.')
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path(COLAB_DIR)
    print('Colab: Drive에 둡니다.')
else:
    # RunPod은 네트워크 볼륨이 /workspace입니다. 없으면(볼륨 없는 팟) 로컬로 떨어지고,
    # 그 사실을 적습니다 - 조용히 로컬에 쓰면 팟을 지울 때 3.7시간이 사라집니다.
    WORK = Path(RUNPOD_DIR)
    anchor = WORK.parent
    if not anchor.exists():
        WORK = Path('.')
        print(f'{anchor}가 없습니다. 이 컨테이너 안에만 씁니다 - **팟을 지우면 사라집니다.**')
        print('RunPod이면 네트워크 볼륨을 /workspace에 붙이고 다시 돌리세요.')
    else:
        print(f'{anchor}에 둡니다(네트워크 볼륨이면 팟을 지워도 남습니다).')

WORK.mkdir(parents=True, exist_ok=True)
SAVE_PATH = WORK / SAVE_DIR
CHUNK_DIR = WORK / 'score_chunks'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
print('작업 폴더:', WORK.resolve())
print('저장 위치:', SAVE_PATH, '| 조각:', CHUNK_DIR)


## 4. 쌍 받기 — 학습용과 평가용

**둘 다 받습니다.** 학습 쌍은 2026-07로 자른 그래프에서, 평가 쌍은 2026-08로 자른
그래프에서 나왔습니다. 섞이면 판정이 성립하지 않습니다.

In [ ]:
import gzip
import json
from pathlib import Path


def fetch(name):
    # 3절에서 받은 토큰을 재사용합니다. 여기서 다시 묻지 않습니다.
    path = Path(name)
    if path.exists():
        return path
    if not USE_HF:
        raise SystemExit(f'{name}이 없습니다. 왼쪽 파일 창에 올리거나 USE_HF=True로 두세요.')
    from huggingface_hub import hf_hub_download
    try:
        return Path(hf_hub_download(f'{HF_OWNER}/{DATA_REPO}', name,
                                    repo_type='dataset', token=HF_TOKEN))
    except Exception as exc:
        # **404를 트레이스백으로 두지 않습니다.** 없는 이유는 거의 항상 "아직 올리지
        # 않았다"이고, 그때 필요한 것은 예외 11겹이 아니라 **다음에 칠 명령**입니다.
        if 'EntryNotFound' not in type(exc).__name__ and '404' not in str(exc):
            raise
        raise SystemExit(
            f'''{name}이 HF 데이터셋({HF_OWNER}/{DATA_REPO})에 없습니다.
저장소에서 만들어 올리세요 — 학습 쌍 둘만 올리면 됩니다
(평가 쌍 rerank_pairs/texts는 이미 올라가 있을 수 있습니다):

  uv run python -m evals.export_rerank_training --max-seeds 1000
  hf upload {HF_OWNER}/{DATA_REPO} \
      exports/colab/{name} {name} --repo-type dataset
'''
        ) from None


def read_jsonl(path):
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle]


train_rows = read_jsonl(fetch(TRAIN_PAIRS))
train_texts = {r['arxiv_id']: r['text'] for r in read_jsonl(fetch(TRAIN_TEXTS))}
eval_rows = read_jsonl(fetch(EVAL_PAIRS))
eval_texts = {r['arxiv_id']: r['text'] for r in read_jsonl(fetch(EVAL_TEXTS))}

if SMOKE:
    train_rows, eval_rows = train_rows[:40], eval_rows[:40]

# 텍스트가 없으면 멈춥니다. 빈 문자열로 학습하거나 채점하면 그것이 모델의 사실처럼
# 보고서에 들어갑니다.
for label, rows, texts, keys in (
    ('학습', train_rows, train_texts, ('positives', 'negatives')),
    ('평가', eval_rows, eval_texts, ('candidates',)),
):
    missing = {p for row in rows
               for p in [row['seed'], *(x for k in keys for x in row[k])]
               if p not in texts}
    if missing:
        raise SystemExit(f'{label} 텍스트가 없는 논문 {len(missing)}편: {sorted(missing)[:5]}')

n_pos = sum(len(r['positives']) for r in train_rows)
n_neg = sum(len(r['negatives']) for r in train_rows)
print(f'학습 시드 {len(train_rows)}개 / 양성 {n_pos} / 음성 {n_neg}  (1:{n_neg / max(1, n_pos):.1f})')
print(f'평가 시드 {len(eval_rows)}개 / 쌍 {sum(len(r["candidates"]) for r in eval_rows)}개')


## 5. 미세조정

**손실이 내려가는지 봅니다.** ADR-057이 적은 선 — *"한 에폭 뒤 학습 손실이 내려가지
않으면 그것이 답이고, 더 돌리지 않습니다."* 그래서 시드의 10%를 검증으로 떼고, 에폭마다
**실측 시간과 VRAM 피크**를 찍습니다(2026-10-20에 어림을 두 번 틀렸습니다).

In [ ]:
import random
import time
from pathlib import Path

import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

_rng = random.Random(20261021)
_seeds = sorted(r['seed'] for r in train_rows)
_rng.shuffle(_seeds)
_cut = max(1, int(len(_seeds) * VAL_RATIO))
val_seeds = set(_seeds[:_cut])


# **시드 단위로 나눕니다.** 쌍 단위로 나누면 같은 시드가 양쪽에 들어가 검증이 샙니다.
def samples_of(rows, keep):
    out = []
    for row in rows:
        if (row['seed'] in val_seeds) != keep:
            continue
        head = train_texts[row['seed']]
        for paper in row['positives']:
            out.append(InputExample(texts=[head, train_texts[paper]], label=1.0))
        for paper in row['negatives']:
            out.append(InputExample(texts=[head, train_texts[paper]], label=0.0))
    return out


train_samples = samples_of(train_rows, keep=False)
val_samples = samples_of(train_rows, keep=True)
_rng.shuffle(train_samples)
print(f'학습 {len(train_samples)}쌍 / 검증 {len(val_samples)}쌍 '
      f'(검증 시드 {len(val_seeds)}개)')

# **중간 곡선은 평가기로 받습니다.** 에폭을 쪼개 fit을 여러 번 부르면 워밍업과 LR 스케줄이
# 매번 다시 시작해 **한 번에 돌린 것과 다른 학습**이 됩니다. 기록을 얻으려고 학습을
# 왜곡하지 않습니다.
_evaluator = None
if val_samples and EVAL_STEPS:
    _evaluator = CEBinaryClassificationEvaluator.from_input_examples(
        val_samples, name='val'
    )

# **저장된 모델이 있으면 학습을 건너뜁니다.** 채점 도중에 끊겼을 때 3.7시간을 다시
# 치르지 않게 하는 장치입니다.
_saved = SAVE_PATH.exists() and any(SAVE_PATH.iterdir())
if RESUME_FROM_SAVED and _saved:
    print(f'{SAVE_PATH}에 저장된 모델이 있어 학습을 건너뜁니다.')
    print('다시 학습하려면 RESUME_FROM_SAVED=False로 두거나 그 폴더를 지우세요.')
    model = CrossEncoder(str(SAVE_PATH), num_labels=1, max_length=MAX_SEQ)
    elapsed, peak, rate, batch = 0.0, 0.0, 0.0, BATCH
    _eval_dir = Path('ce_eval')
else:
    model = CrossEncoder(MODEL, num_labels=1, max_length=MAX_SEQ)
    _eval_dir = Path('ce_eval')

    batch = BATCH
    while True:
        try:
            loader = DataLoader(train_samples, shuffle=True, batch_size=batch)
            started = time.perf_counter()
            torch.cuda.reset_peak_memory_stats()
            model.fit(
                train_dataloader=loader,
                evaluator=_evaluator,
                evaluation_steps=EVAL_STEPS if _evaluator else 0,
                output_path=str(_eval_dir) if _evaluator else None,
                epochs=EPOCHS,
                warmup_steps=int(len(loader) * EPOCHS * WARMUP_RATIO),
                optimizer_params={'lr': LR},
                show_progress_bar=True,
            )
            break
        except torch.cuda.OutOfMemoryError:
            # **터지는 대신 느려집니다.** 2026-10-20에 배치 256이 OOM이고 128로 완주했습니다.
            torch.cuda.empty_cache()
            if batch <= 1:
                raise SystemExit('배치 1에서도 OOM입니다. MAX_SEQ를 줄이거나 큰 GPU가 필요합니다.')
            batch //= 2
            print(f'OOM. 배치를 {batch}로 줄여 다시 시작합니다.')

    elapsed = time.perf_counter() - started
    peak = torch.cuda.max_memory_allocated() / 1e9
    rate = len(train_samples) * EPOCHS / elapsed
    # **학습이 끝나면 먼저 저장합니다.** 채점은 다시 돌릴 수 있지만 학습은 3.7시간입니다.
    model.save(str(SAVE_PATH))
    print('저장:', SAVE_PATH)
print(f'\n학습 끝: {elapsed / 60:.1f}분, 배치 {batch}, VRAM 피크 {peak:.1f}GB, {rate:.1f}쌍/초')


### MLflow에 남깁니다 — 끊겨도 남는 기록으로

ADR-057의 바와 나란히 놓을 수 있어야 합니다. **점수 파일(11MB)은 올리지 않습니다** —
그것은 HF에 두고 여기에는 **요약만** 둡니다. 두 곳에 두면 둘이 갈라집니다.


In [ ]:
import csv
import statistics


def _val_scores():
    """검증 쌍의 양성·음성 점수. 판정이 아니라 "학습이 무엇도 안 했는지"의 신호입니다."""
    if not val_samples:
        return None, None
    pairs_in = [(s.texts[0], s.texts[1]) for s in val_samples]
    labels = [s.label for s in val_samples]
    values = model.predict(pairs_in, batch_size=batch, show_progress_bar=True)
    pos = [v for v, y in zip(values, labels) if y == 1.0]
    neg = [v for v, y in zip(values, labels) if y == 0.0]
    return pos, neg


pos, neg = _val_scores()
summary = {
    'train_minutes': round(elapsed / 60, 2),
    'train_pairs_per_second': round(rate, 2),
    'vram_peak_gb': round(peak, 2),
    'batch_used': batch,
    'train_pairs': len(train_samples),
    'val_pairs': len(val_samples),
}
if pos and neg:
    _rng2 = random.Random(7)
    wins = sum(1 for _ in range(20000) if _rng2.choice(pos) > _rng2.choice(neg))
    summary['val_pos_median'] = round(statistics.median(pos), 4)
    summary['val_neg_median'] = round(statistics.median(neg), 4)
    # 쌍을 무작위로 하나씩 뽑아 양성이 더 높은 비율. 0.5면 아무 일도 안 한 것입니다.
    summary['val_auc_approx'] = round(wins / 20000, 4)

for key, value in summary.items():
    print(f'  {key}: {value}')

if mlflow is not None:
    with mlflow.start_run(run_name=f'rerank-cocite-ft-{len(train_rows)}seeds'):
        mlflow.log_params({
            'base_model': MODEL,
            'max_seq': MAX_SEQ,
            'batch_requested': BATCH,
            'batch_used': batch,
            'lr': LR,
            'epochs': EPOCHS,
            'warmup_ratio': WARMUP_RATIO,
            'val_ratio': VAL_RATIO,
            'train_seeds': len(train_rows),
            'val_seeds': len(val_seeds),
            'positives': sum(len(r['positives']) for r in train_rows),
            'negatives': sum(len(r['negatives']) for r in train_rows),
            # **자르는 선을 적습니다.** 이것이 없으면 그 run이 누출된 데이터로 학습한
            # 것인지 나중에 알 수 없습니다(ADR-057).
            'train_cut': '2026-07',
            'test_cut': '2026-08',
            'smoke': SMOKE,
        })
        mlflow.log_metrics(summary)
        # 평가기가 남긴 중간 곡선을 그대로 올립니다. 판마다 열 이름이 달라서 수치 열만
        # 골라 스텝으로 찍습니다 - 모양을 가정하지 않습니다.
        for _csv in sorted(_eval_dir.rglob('*.csv')) if _eval_dir.exists() else []:
            with _csv.open(encoding='utf-8') as handle:
                for step, line in enumerate(csv.DictReader(handle)):
                    for key, raw in line.items():
                        try:
                            mlflow.log_metric(f'val_{key}', float(raw), step=step)
                        except (TypeError, ValueError):
                            continue
            mlflow.log_artifact(str(_csv))
        # 내보내기 매니페스트가 있으면 함께 둡니다 - 누출 검사가 그 안에 있습니다.
        for _m in ('rerank_training_manifest.json',):
            if Path(_m).exists():
                mlflow.log_artifact(_m)
        print('MLflow run 기록 끝:', mlflow.active_run().info.run_id)
else:
    print('추적이 꺼져 있어 이 요약은 출력에만 남습니다.')


### 검증 쌍에서 양성과 음성이 갈리는가

**이것이 "더 돌릴 가치가 있는가"의 신호입니다.** 판정이 아닙니다 — 판정은 저장소가 간선
0개 집단에서 냅니다. 여기서 보는 것은 **학습이 아무 일도 하지 않았는지**뿐입니다.

In [ ]:
import statistics

if val_samples:
    _pairs = [(s.texts[0], s.texts[1]) for s in val_samples]
    _labels = [s.label for s in val_samples]
    _scores = model.predict(_pairs, batch_size=batch, show_progress_bar=True)
    pos = [s for s, y in zip(_scores, _labels) if y == 1.0]
    neg = [s for s, y in zip(_scores, _labels) if y == 0.0]
    if pos and neg:
        print(f'검증 양성 중앙값 {statistics.median(pos):.4f}  '
              f'음성 중앙값 {statistics.median(neg):.4f}')
        # 쌍을 무작위로 하나씩 뽑아 양성이 더 높은 비율(AUC와 같은 양).
        _rng2 = random.Random(7)
        wins = sum(1 for _ in range(20000)
                   if _rng2.choice(pos) > _rng2.choice(neg))
        print(f'양성이 음성보다 높을 확률 ~ {wins / 20000:.3f}  (0.5면 아무 일도 안 한 것)')
    else:
        print('검증에 한쪽 라벨만 있습니다. VAL_RATIO를 올리세요.')
else:
    print('검증 쌍이 없습니다. VAL_RATIO를 올리세요.')

## 6. 평가 쌍 채점 — 저장소가 판정할 수 있는 모양으로

출력 모양을 **사전학습 노트북과 같게** 둡니다. 그래야 저장소가 **같은 명령**으로 판정하고,
같은 후보·같은 정답·같은 실행이 됩니다.

In [ ]:
import re
import time

import torch

_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-') + '-cocite-ft'
out = Path(f'rerank-{_slug}.jsonl.gz')

# **조각으로 씁니다.** 끊겼다 다시 돌리면 이미 끝난 조각을 건너뜁니다 - 사전학습
# 노트북이 쓰는 것과 같은 방식입니다(rerank_pairs_colab).
batch_eval = batch or BATCH
started, done = time.perf_counter(), 0
for begin in range(0, len(eval_rows), CHUNK_SEEDS):
    part = CHUNK_DIR / f'{begin:06d}.jsonl'
    if part.exists():
        continue
    block = eval_rows[begin:begin + CHUNK_SEEDS]
    written = []
    for row in block:
        head = eval_texts[row['seed']]
        pairs_in = [(head, eval_texts[c]) for c in row['candidates']]
        while True:
            try:
                torch.cuda.reset_peak_memory_stats()
                values = model.predict(pairs_in, batch_size=batch_eval)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if batch_eval <= 1:
                    raise SystemExit('배치 1에서도 OOM입니다.')
                batch_eval //= 2
                print(f'OOM. 채점 배치를 {batch_eval}로 줄입니다.')
        # **모양을 판정기에 맞춥니다.** `evals.related_study.RerankScores`는 줄마다
        # `candidates`(목록)와 `scores`(목록)를 색인으로 짝지어 읽습니다.
        written.append(json.dumps({
            'seed': row['seed'],
            'candidates': row['candidates'],
            'scores': [round(float(v), 6) for v in values],
        }, ensure_ascii=False))
        done += len(pairs_in)
    # **조각을 다 쓴 뒤에 옮깁니다.** 쓰는 중에 끊기면 반쪽 조각이 "끝난 것"으로 보입니다.
    tmp = part.with_suffix('.part')
    tmp.write_text('\n'.join(written) + '\n', encoding='utf-8')
    tmp.replace(part)
    took = time.perf_counter() - started
    peak_eval = torch.cuda.max_memory_allocated() / 1e9
    print(f'  조각 {part.name}  누적 {done}쌍  {took / 60:.1f}분  '
          f'{done / max(took, 1e-9):.1f}쌍/초  배치 {batch_eval}  VRAM {peak_eval:.1f}GB')

# 조각을 하나로 모읍니다. 시드 수가 평가 쌍 수와 같아야 합니다.
seen = set()
with gzip.open(out, 'wt', encoding='utf-8', newline='\n') as handle:
    for part in sorted(CHUNK_DIR.glob('*.jsonl')):
        for line in part.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            seen.add(json.loads(line)['seed'])
            handle.write(line + '\n')

# **빠진 시드가 있으면 멈춥니다.** 잘린 파일로 판정하면 그 수가 모델의 사실처럼 보입니다.
_missing = {row['seed'] for row in eval_rows} - seen
if _missing:
    raise SystemExit(
        f'시드 {len(_missing)}개가 빠졌습니다(예: {sorted(_missing)[:3]}). '
        f'{CHUNK_DIR}를 지우지 않고 이 셀을 다시 돌리면 빠진 조각만 채웁니다.'
    )

print(f'\n{out.name}  {out.stat().st_size / 1e6:.1f}MB  시드 {len(seen)}개')


## 7. 저장하고 올리기

In [ ]:
import re
from pathlib import Path

from huggingface_hub import HfApi

# `MODEL`에서 이름을 다시 만듭니다 - 채점이 끝난 뒤 커널이 끊겨도 1·3절과 이 셀만으로
# 올라가야 합니다.
_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-') + '-cocite-ft'
_target = Path(f'rerank-{_slug}.jsonl.gz')
if not _target.exists():
    raise SystemExit(f'{_target.name}이 없습니다. 6절을 먼저 돌리세요.')
print(f'{_target.name}  {_target.stat().st_size / 1e6:.1f}MB')

_api = HfApi(token=HF_TOKEN)
_repo = f"{HF_OWNER}/{DATA_REPO}"
_api.create_repo(_repo, repo_type='dataset', private=True, exist_ok=True)
_api.upload_file(path_or_fileobj=str(_target), path_in_repo=_target.name,
                 repo_id=_repo, repo_type='dataset')
print('올림:', _target.name)

if UPLOAD_WEIGHTS:
    # 2.3GB입니다. 다시 학습하지 않으려면 올려 두는 편이 낫고, 점수만 필요하면 끄세요.
    _model_repo = f"{HF_OWNER}/{DATA_REPO}-reranker-cocite-ft"
    model.save(str(Path('finetuned')))
    _api.create_repo(_model_repo, private=True, exist_ok=True)
    _api.upload_folder(folder_path='finetuned', repo_id=_model_repo)
    print('가중치 올림:', _model_repo)

print(f'\nrepo: {_repo}')
print('저장소에서 받아 판정할 명령:')
print(f'  hf download {_repo} {_target.name} \\')
print('      --repo-type dataset --local-dir derived/2026-10-14')
print('  uv run python -m evals.related_study --cases evals/cases_related_future.json \\')
print('      --variants rerank vector \\')
print(f'      --rerank-scores derived/2026-10-14/{_target.name} \\')
print('      --output reports/rerank_cocite_ft.json')